# Q-Mini — Initial Model Builder (GitHub → Colab → Google Drive)

This build-only notebook fetches the current project from GitHub, builds the untrained 0.7M-class Q2-v2 initial model in Colab, and saves the model plus the complete project source and disabled training runtime to Google Drive.

- No dataset upload is needed to build the initial model.
- Context length is fixed at 512 tokens.
- `project_source/` contains the GitHub checkout (configs, scripts, tests, docs and notebooks), excluding Git history and temporary caches.
- `training_runtime/` contains the matching model/training/checkpoint source and disabled training defaults.
- Tokenizer and prepared train/validation streams are not created in this initial-build stage; they are required before actual training.
- Colab's `/content` folder is temporary; the verified package is saved in Google Drive.

Run cells from top to bottom. Existing Drive builds are never overwritten.

In [ ]:
# -------- Build settings (edit before running) --------
MODEL_CONFIG = "configs/qmini_light_0.7m.json"
DRIVE_SUBFOLDER = "QMini-Light-0.7M/initial_brains"
REPOSITORY_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"


print("Selected model configuration:", MODEL_CONFIG)
print("Build mode: INITIALIZATION ONLY — TRAINING DISABLED")

In [ ]:
# -------- Mount Drive and fetch a fresh source checkout from GitHub --------from google.colab import drivedrive.mount("/content/drive", force_remount=False)from pathlib import Pathfrom datetime import datetime, timezoneimport hashlibimport jsonimport osimport shutilimport subprocessimport sysimport timeimport uuidDRIVE_ROOT = Path("/content/drive/MyDrive")if not DRIVE_ROOT.is_dir():    raise RuntimeError("Google Drive was not mounted at /content/drive/MyDrive.")RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]REPO_DIR = Path("/content") / f"Q-Mini-{RUN_ID}"DRIVE_PARENT = DRIVE_ROOT / DRIVE_SUBFOLDERDRIVE_PARENT.mkdir(parents=True, exist_ok=True)def run_checked(command, *, cwd=None, capture_output=False, env=None):    command = [str(part) for part in command]    print("$", " ".join(command))    return subprocess.run(        command, cwd=str(cwd) if cwd else None, check=True,        text=True, capture_output=capture_output, env=env,    )# Public repository: clone the latest main branch into a new folder every run.# This avoids stale or missing /content paths after a Colab runtime restart.run_checked([    "git", "clone", "--depth", "1", "--branch", "main",    REPOSITORY_URL, REPO_DIR,])SOURCE_REVISION = run_checked(    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True).stdout.strip()print("GitHub revision:", SOURCE_REVISION)print("Fresh source checkout:", REPO_DIR)print("Drive destination:", DRIVE_PARENT)

In [ ]:
# -------- Install project without replacing Colab's PyTorch wheel --------run_checked([    sys.executable, "-m", "pip", "install",    "numpy>=1.24", "tokenizers>=0.15", "packaging>=23",])run_checked([    sys.executable, "-m", "pip", "install",    "--no-deps", "-e", ".",], cwd=REPO_DIR)import torchfrom packaging.version import Versiontorch_version = Version(torch.__version__.split("+", 1)[0])if torch_version < Version("2.10.0"):    raise RuntimeError(        f"Q-Mini requires PyTorch >=2.10.0; Colab has {torch.__version__}. "        "Select a compatible Colab runtime. PyTorch will not be replaced blindly."    )print("Python:", sys.version.split()[0])print("PyTorch:", torch.__version__)print("CUDA build:", torch.version.cuda)print("CUDA available:", torch.cuda.is_available())if torch.cuda.is_available():    print("GPU:", torch.cuda.get_device_name(0))else:    print("Initial Q2-v2 build is CPU-streamed; no GPU is required for this stage.")

In [ ]:
# -------- Build the untrained Q2-v2 initial model locally --------
# This command calls only qmini build-initial. It does not invoke qmini train.
LOCAL_BUILD_DIR = Path("/content") / f"QMini-initial-{RUN_ID}"
if LOCAL_BUILD_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite local build directory: {LOCAL_BUILD_DIR}")

BUILD_STARTED_UTC = datetime.now(timezone.utc).isoformat()
build_clock = time.monotonic()

run_checked([
    sys.executable, "-m", "qmini.cli", "build-initial",
    "--config", MODEL_CONFIG,
    "--output-dir", LOCAL_BUILD_DIR,
    "--source-revision", SOURCE_REVISION,
], cwd=REPO_DIR)

BUILD_SECONDS = round(time.monotonic() - build_clock, 2)
PACKED_PATH = LOCAL_BUILD_DIR / "model.qmini.npz"
MANIFEST_PATH = LOCAL_BUILD_DIR / "model_manifest.json"
CONFIG_OUT_PATH = LOCAL_BUILD_DIR / "config.json"

for required_file in (PACKED_PATH, MANIFEST_PATH, CONFIG_OUT_PATH):
    if not required_file.is_file():
        raise FileNotFoundError(f"Builder did not produce required file: {required_file}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert manifest.get("format") == "qmini-initial-brain-v1"
assert manifest.get("status") == "untrained_initialization"
assert manifest.get("training_steps") == 0
assert manifest.get("optimizer_steps") == 0
assert manifest.get("tokens_seen") == 0
assert manifest.get("training_started") is False
assert manifest.get("matrix_weight_format") == "qmini-packed-v2"
assert manifest.get("matrix_weight_bits") == 2
assert manifest.get("parameters") == cfg.estimated_parameter_count()
assert not (LOCAL_BUILD_DIR / "model.pt").exists(), "Unexpected dense training checkpoint."

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8*1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

LOCAL_SHA256 = sha256_file(PACKED_PATH)
assert LOCAL_SHA256 == manifest.get("packed_export_sha256"), "Local artifact SHA-256 mismatch."

verify_proc = run_checked(
    [sys.executable, "-m", "qmini.cli", "verify-export", "--path", PACKED_PATH],
    cwd=REPO_DIR,
    capture_output=True,
)
PACKED_REPORT = json.loads(verify_proc.stdout)
assert PACKED_REPORT.get("format") == "qmini-packed-v2"
assert PACKED_REPORT.get("q2_version") == 2
assert PACKED_REPORT.get("parameter_count") == cfg.estimated_parameter_count()

print("Local initial model built and independently verified.")
print("Parameters:", f"{cfg.estimated_parameter_count():,}")
print("Packed artifact size:", f"{PACKED_PATH.stat().st_size:,}", "bytes")
print("SHA-256:", LOCAL_SHA256)
print("Build time:", BUILD_SECONDS, "seconds")
print("Training steps: 0 | Optimizer steps: 0 | Tokens seen: 0")

In [ ]:
# -------- Copy to Google Drive, verify the copy, then publish --------FINAL_DIR = DRIVE_PARENT / f"QMini-{Path(MODEL_CONFIG).stem}-Q2v2-initial-{RUN_ID}"STAGING_DIR = DRIVE_PARENT / f".{FINAL_DIR.name}.staging"if FINAL_DIR.exists():    raise FileExistsError(f"Refusing to overwrite existing Drive build: {FINAL_DIR}")if STAGING_DIR.exists():    raise FileExistsError(f"Refusing to overwrite existing staging directory: {STAGING_DIR}")STAGING_DIR.mkdir(parents=False, exist_ok=False)for filename in ("config.json", "model.qmini.npz", "model_manifest.json"):    shutil.copy2(LOCAL_BUILD_DIR / filename, STAGING_DIR / filename)DRIVE_PACKED_PATH = STAGING_DIR / "model.qmini.npz"DRIVE_SHA256 = sha256_file(DRIVE_PACKED_PATH)if DRIVE_SHA256 != LOCAL_SHA256:    raise RuntimeError("Drive copy SHA-256 differs from the verified local artifact.")drive_verify_proc = run_checked(    [sys.executable, "-m", "qmini.cli", "verify-export", "--path", DRIVE_PACKED_PATH],    cwd=REPO_DIR,    capture_output=True,)DRIVE_PACKED_REPORT = json.loads(drive_verify_proc.stdout)if DRIVE_PACKED_REPORT.get("parameter_count") != cfg.estimated_parameter_count():    raise RuntimeError("Drive copy parameter count does not match the selected config.")drive_manifest = json.loads((STAGING_DIR / "model_manifest.json").read_text(encoding="utf-8"))if drive_manifest.get("packed_export_sha256") != DRIVE_SHA256:    raise RuntimeError("Drive manifest fingerprint does not match the copied artifact.")# -------- Bundle complete GitHub source and dormant training code --------PROJECT_SOURCE_DIR = STAGING_DIR / "project_source"shutil.copytree(    REPO_DIR,    PROJECT_SOURCE_DIR,    ignore=shutil.ignore_patterns(        ".git", "__pycache__", "*.pyc", ".pytest_cache",        ".venv", "venv", ".mypy_cache", ".ruff_cache",    ),)RUNTIME_DIR = STAGING_DIR / "training_runtime"RUNTIME_DIR.mkdir()shutil.copytree(    REPO_DIR / "qmini",    RUNTIME_DIR / "qmini",    ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.so", "*.pyd"),)shutil.copy2(REPO_DIR / "pyproject.toml", RUNTIME_DIR / "pyproject.toml")TRAINING_DEFAULTS = {    "format": "qmini-training-defaults-v1",    "enabled": False,    "sequence_length": 512,    "batch_size": 1,    "gradient_accumulation": 8,    "learning_rate": 3e-4,    "weight_decay": 0.1,    "checkpoint_every_optimizer_steps": 10,    "evaluation_every_optimizer_steps": 100,    "total_optimizer_steps": 20000,    "minimum_free_gpu_gib": 8.0,}(RUNTIME_DIR / "TRAINING_DEFAULTS.json").write_text(    json.dumps(TRAINING_DEFAULTS, indent=2, sort_keys=True) + "\n",    encoding="utf-8",)(RUNTIME_DIR / "README.md").write_text(    "# Q-Mini training source bundle\n\n"    "Exact qmini source from the GitHub revision in the initial model manifest. "    "Training is disabled. This initial package does not yet include a tokenizer "    "or prepared train/validation streams; those are required before actual training.\n",    encoding="utf-8",)runtime_source_files = []for source_file in sorted(    path for path in RUNTIME_DIR.rglob("*")    if path.is_file() and path.name != "TRAINING_RUNTIME.json"):    runtime_source_files.append({        "path": source_file.relative_to(RUNTIME_DIR).as_posix(),        "sha256": sha256_file(source_file),        "size_bytes": source_file.stat().st_size,    })RUNTIME_MANIFEST = {    "format": "qmini-training-runtime-v1",    "enabled": False,    "requires_manual_activation": True,    "entrypoint": "qmini.train.train_model",    "checkpoint_backend": "qmini.checkpointing",    "source_revision": SOURCE_REVISION,    "source_files": runtime_source_files,}RUNTIME_MANIFEST_PATH = RUNTIME_DIR / "TRAINING_RUNTIME.json"RUNTIME_MANIFEST_PATH.write_text(    json.dumps(RUNTIME_MANIFEST, indent=2, sort_keys=True) + "\n",    encoding="utf-8",)STATE_DIR = STAGING_DIR / "training_state"STATE_DIR.mkdir()(STATE_DIR / "training_progress.json").write_text(    json.dumps({        "format": "qmini-training-progress-v1",        "status": "not_started",        "current_step": 0,        "target_steps": None,        "progress_percent": 0.0,        "tokens_seen": 0,        "updated_at": datetime.now(timezone.utc).isoformat(),    }, indent=2, sort_keys=True) + "\n",    encoding="utf-8",)drive_manifest["training_runtime_included"] = Truedrive_manifest["training_runtime_enabled"] = Falsedrive_manifest["training_runtime_manifest_sha256"] = sha256_file(RUNTIME_MANIFEST_PATH)(STAGING_DIR / "model_manifest.json").write_text(    json.dumps(drive_manifest, indent=2, sort_keys=True) + "\n",    encoding="utf-8",)print("GitHub source files saved:", sum(1 for p in PROJECT_SOURCE_DIR.rglob("*") if p.is_file()))print("Training/checkpoint source files saved:", len(runtime_source_files))print("Training enabled:", False)BUILD_REPORT = {    "format": "qmini-colab-build-report-v1",    "run_id": RUN_ID,    "repository": REPOSITORY_URL,    "source_revision": SOURCE_REVISION,    "config": MODEL_CONFIG,    "parameters": cfg.estimated_parameter_count(),    "architecture": {        "causal_convolution": True,        "diagonal_state_space": bool(cfg.use_state_space),        "sparse_top1_moe": True,        "transformer_self_attention": False,        "qkv_attention": False,    },    "artifact": {        "filename": "model.qmini.npz",        "format": "qmini-packed-v2",        "matrix_weight_bits": 2,        "bytes": DRIVE_PACKED_PATH.stat().st_size,        "sha256": DRIVE_SHA256,        "verified_after_drive_copy": True,        "verification": DRIVE_PACKED_REPORT,    },    "training": {        "started": False,        "training_steps": 0,        "optimizer_steps": 0,        "tokens_seen": 0,        "tokenizer_included": False,    },    "preflight": preflight,    "build_started_utc": BUILD_STARTED_UTC,    "build_completed_utc": datetime.now(timezone.utc).isoformat(),    "build_seconds": BUILD_SECONDS,    "project_source_directory": "project_source/",    "training_runtime_directory": "training_runtime/",    "training_runtime_source_files": len(runtime_source_files),    "tokenizer_prepared": False,    "dataset_prepared": False,}(STAGING_DIR / "BUILD_REPORT.json").write_text(    json.dumps(BUILD_REPORT, indent=2, ensure_ascii=False, sort_keys=True) + "\n",    encoding="utf-8",)# Publish only after the Drive copy and its Q2 archive have passed verification.STAGING_DIR.rename(FINAL_DIR)print("\nQ-MINI INITIAL BUILD COMPLETE")print("Drive folder:", FINAL_DIR)print("Model artifact:", FINAL_DIR / "model.qmini.npz")print("Build report:", FINAL_DIR / "BUILD_REPORT.json")print("SHA-256:", DRIVE_SHA256)print("TRAINING STARTED: NO")print("This is a verified, untrained initial model. Training must be run as a separate stage.")

## What is saved to Google Drive

- `config.json`, `model.qmini.npz`, and `model_manifest.json` — verified initial Q2-v2 model.
- `project_source/` — the complete GitHub source revision used for this build, including configs, scripts, tests, docs and Colab notebooks (excluding `.git` history and temporary caches).
- `training_runtime/` — the matching `qmini` model/training/checkpoint source, dependency metadata, runtime manifest and disabled training defaults.
- `training_state/training_progress.json` — starts at `not_started`, step zero.
- `BUILD_REPORT.json` — build and verification report.

This initial build needs no corpus and does not start training. The weights are random initialization, not a trained assistant. A tokenizer and matching train/validation data streams must be prepared before training can begin.